### <u> Generate a local "_Stars Appearing_" sequence: sonification + animation </u>

This builds the "_Stars Appearing_" piece from the "_Audible Universe_" planetarium
show for **any site and any night**, and renders a matching animation to go with it.

The sky is computed with `skyfield` from the _Hipparcos_ catalogue, sonified with
`strauss`, and animated as an **equirectangular** (360&deg; &times; 180&deg;)
panorama. A planetarium dome master comes from letting `ffmpeg`'s `v360` filter
reproject that panorama to fisheye &mdash; we never render fisheye ourselves.

The important idea: the animation does **not** work out for itself when each star
appears. It reads the timings straight out of the rendered sonification, via
`Sonification.event_table()`, so sound and picture cannot drift apart.

All the machinery lives in `StarsAppearingLocal.py` alongside this notebook, so
there is one copy of it rather than two. This notebook drives it.

#### Requirements

Beyond `strauss` itself you need `skyfield`, and a working `ffmpeg` on your `PATH`.
The first run downloads the _Hipparcos_ catalogue (~50 MB) and the DE421 ephemeris
(~17 MB) into a cache directory; later runs reuse them.

In [ ]:
# %pip --quiet install skyfield

In [ ]:
import matplotlib.pyplot as plt

from StarsAppearingLocal import (Config, observed_sky, build_sonification,
                                 timings, write_videos, make_sequence)

### <u> Chosen properties </u>

Change these and re-run. The defaults describe the **Sherwood Observatory** site
looking south, at a small size and with a fast-rendering instrument so the whole
notebook runs in seconds.

&#9999;&#9999;&#9999;&#9999;&#9999;&#9999;&#9999;&#9999;&#9999;&#9999;&#9999;&#9999;

In [ ]:
cfg = Config(
    # -- where and when --
    latitude=53.1143737,          # +ve north
    longitude=-1.2219389,         # +ve *east*, so 1.22 degrees west is -1.22
    date_time="2026-03-25 18:45:00",
    time_zone="Europe/London",
    facing="S",                   # centre of the panorama and of the stereo image
    mag_limit=4.5,                # higher includes more, dimmer stars

    # -- the sound --
    duration=20.0,
    system="stereo",
    instrument="mallets",

    # -- the picture --
    width=640, height=360, fps=12,

    outdir="stars_appearing_preview",
)

# For the real thing, closer to what the planetarium used:
#
#     cfg = Config(mag_limit=5, duration=60, system="5.1",
#                  instrument="glockenspiel",
#                  width=4096, height=2160, fps=30,
#                  background="sherwood_sky.png",
#                  outdir="stars_appearing_full")
#
# `background` is an equirectangular panorama of the site with `facing` at its
# centre; without one the sky is simply black.

&#9999;&#9999;&#9999;&#9999;&#9999;&#9999;&#9999;&#9999;&#9999;&#9999;&#9999;&#9999;

### <u> The sky </u>

`skyfield` gives the altitude and azimuth of every catalogue star as seen from the
chosen site at the chosen instant. The catalogue is cut down by magnitude *before*
positions are computed, which is the difference between transforming ~118,000 stars
and ~1,500.

In [ ]:
sky = observed_sky(cfg)
print(f"{len(sky)} stars brighter than magnitude {cfg.mag_limit} above the horizon")
sky.head()

A quick look at what we are about to hear, as it would appear on the
panorama &mdash; the facing direction in the middle, the horizon along the bottom.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.set_facecolor("#0b0c15")

from StarsAppearingLocal import facing_degrees, CARDINALS
x = (sky["az"] - facing_degrees(cfg.facing) - 180) % 360

ax.scatter(x, sky["alt"], s=40 * 10 ** (-0.2 * sky["magnitude"]),
           c=sky["bv"], cmap="RdYlBu_r", vmin=-0.2, vmax=1.8, lw=0)
ax.set_xticks([(360 * i / 16 - facing_degrees(cfg.facing) - 180) % 360
               for i in range(16)])
ax.set_xticklabels(CARDINALS, fontsize=8)
ax.set_xlim(0, 360)
ax.set_ylim(0, 90)
ax.set_xlabel("compass direction")
ax.set_ylabel("altitude [deg]")
ax.set_title(f"{len(sky)} stars over {cfg.latitude:.2f}, {cfg.longitude:.2f} "
             f"at {cfg.date_time}")
plt.show()

### <u> The sonification </u>

Brightest stars sound first, so magnitude drives `time`. Colour drives `pitch`,
picking a note from the chord: blue stars take high notes and red stars low ones,
carrying the short-to-long wavelength of the light onto the short-to-long
wavelength of the sound. Position drives the
spatial angles, so the sound arrives from where the star is.

A note on angles, since it is easy to get backwards: `strauss` measures azimuth
**anticlockwise from straight ahead**, while astronomical azimuth runs **clockwise
from north**.

In [ ]:
soni = build_sonification(sky, cfg)
soni.notebook_display()

### <u> What sounded, and when </u>

This is the join between sound and picture. `event_table()` reports what is
actually heard &mdash; the time of each note in seconds, the note itself, and the
star's angles in degrees &mdash; and the animation reads its timings from here
rather than recomputing them.

In [ ]:
events = timings(soni, sky)
events.head(10)

### <u> The animation </u>

Each star is a pulse that swells and fades as its note sounds. Frames are generated
as raw `RGBA` and piped straight into `ffmpeg`, which lays them over the background,
muxes the audio, and writes the finished video &mdash; nothing touches the disk in
between.

A star is only drawn while it is bigger than half a pixel, which for these
envelopes is around a second, so only the handful of stars actually alive in each
frame get any work done on them.

The audio is saved with `embed_caption=False`: a caption would be *prepended* to
the track, sliding the whole thing against the picture.

Two videos come out of this, from a single pass of the frame generator:

- the **panorama**, equirectangular, 360&deg; across by 180&deg; high
- the **dome master**, the same pixels reprojected to a 180&deg; fisheye by
  `ffmpeg`'s `v360` filter and tilted so the zenith lands in the centre of the
  dome. The facing direction ends up at the bottom of the image, which is the
  usual dome-master orientation.

They are written to disk rather than shown inline. Embedding a video in the
notebook base64-encodes the whole file into the output, which bloats the `.ipynb`
by a third more than the video itself &mdash; and notebook viewers often refuse to
play the audio of an embedded `data:` URI anyway. Open the files in a normal video
player to check them.

In [ ]:
audio = cfg.outdir / f"stars_appearing_{cfg.instrument}.wav"
cfg.outdir.mkdir(parents=True, exist_ok=True)
soni.save(str(audio), embed_caption=False)

panorama, dome = write_videos(cfg, events, audio, [
    (cfg.outdir / "stars_appearing_panorama.mp4", False),
    (cfg.outdir / "stars_appearing_dome.mp4", True),
])

In [ ]:
for path in (audio, panorama, dome):
    print(f"{path.stat().st_size / 1e6:8.1f} MB  {path}")

### <u> All of it at once </u>

`make_sequence` does every step above in order and returns the paths it wrote, which
is what you want once you have settled on your settings. It also renders the frames
just **once** and feeds both outputs from that single pass, rather than drawing
everything twice:

```python
make_sequence(cfg)
```

Rendering time is dominated by frame size. The preview settings here take a few
seconds. A full 4096 &times; 2160, 60-second sequence at 30 fps in `5.1`, giving both
the panorama and the dome master, takes a couple of minutes on a modern laptop.
Render at preview size while you are still choosing a site, a night and an
instrument, then raise `width`, `height` and `fps` for the final pass.